# Degree Zero — chain notebook (T0 → S1 → T1, vLLM)

One round trip through translation for every original text in the chosen Ship of Theseus domains, for each `LANGS × MODELS`. The same model translates in both directions.

| code | what it is | where |
|---|---|---|
| **G** | word-for-word MUSE gloss (no model) | zero point for S1 (Spanish only for now) |
| **F** | free translation EN → target | S1, all items |
| **F2** | free, reworded prompt | noise floor, `NOISE_N` items per domain × source |
| **L** | literal translation (optional, `RUN_LITERAL`) | S1, second intervention strength |
| **B_F / B_L** | F / L translated back to English | T1 |

Generation only. Metrics (int_G, QE, Burrows' Delta, …) come from the metrics notebook / `round_trip_stats.py`.

**Checkpointing:** every finished chunk is appended to `generations.jsonl` on Drive. If Colab disconnects, re-run all cells; finished generations are skipped.

**Runtime → Change runtime type → GPU** (T4 works; L4/A100 is 2–4× faster).

In [ ]:
!pip install -q vllm sacrebleu
import os, re, json, time, gc, numpy as np, pandas as pd, torch, vllm
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — switch runtime to GPU")
print("vllm", vllm.__version__, "| torch", torch.__version__)
# If pip upgraded torch and Colab asks to restart the session: restart, then run from the next cell.

## Config

In [ ]:
DOMAINS  = ["xsum"]                          # all: cmv eli5 sci_gen tldr wp xsum yelp
SOURCES  = None                              # None = all 7 (Human, OpenAI, LLAMA, Tsinghua, BigScience, PaLM, Eleuther-AI)
LANGS    = ["es"]                            # "zh" still needs its gloss + output filter
MODELS   = ["Qwen/Qwen3-4B-Instruct-2507"]
MAX_W    = None                              # None = every item; 110 = main-study cap (COMETKiwi truncation)
SMOKE_N  = None                              # 50 = quick check on 50 items; None = full run
RUN_LITERAL = True                           # L and B_L: second intervention strength (doubles generation time)
NOISE_N  = 20                                # F2 items per domain × source

MAX_MODEL_LEN = 8192
GPU_MEM  = 0.90
DTYPE    = "half"                            # T4 needs "half"; on L4/A100 "bfloat16" suits bf16-trained models (Gemma, Llama)
CHUNK    = 256                               # prompts per vLLM call; each chunk is checkpointed
OUT_DIR  = "/content/drive/MyDrive/degree_zero"
SEED     = 0

## 1 · Items (T0)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
try:                                         # gated models (Llama, Gemma) need an HF_TOKEN in Colab Secrets
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass
os.makedirs(OUT_DIR, exist_ok=True)

BASE = ("https://raw.githubusercontent.com/tripto03/Ship_of_theseus_paraphrased_copus/"
        "main/paraphrased_datasets/{}_paraphrased.csv")
frames = []
for d in DOMAINS:
    x = pd.read_csv(BASE.format(d))
    x = x[x.version_name == "original"].drop_duplicates(["key", "source"]).assign(domain=d)
    frames.append(x[["domain", "source", "key", "text"]])
items = pd.concat(frames).rename(columns={"text": "S"}).dropna(subset=["S"])
items["words"] = items.S.str.split().str.len()
if SOURCES:
    items = items[items.source.isin(SOURCES)]
if MAX_W:
    items = items[items.words <= MAX_W]
items["item_id"] = items.domain + "|" + items.source + "|" + items.key
items = items.sort_values(["domain", "source", "key"]).reset_index(drop=True)
if SMOKE_N:
    items = items.sample(min(SMOKE_N, len(items)), random_state=SEED).sort_values("item_id").reset_index(drop=True)

# noise-floor subset: same items for every model and language
noise_ids = set(items.sample(frac=1, random_state=SEED).groupby(["domain", "source"]).head(NOISE_N).item_id)

print(len(items), "items |", items.words.sum(), "words | median", items.words.median(),
      "| p95", items.words.quantile(.95), "| F2 items:", len(noise_ids))
display(items.groupby(["domain", "source"]).size().unstack(0))

## 2 · Gloss baseline G (Spanish, deterministic)
Same gloss as the pilot: MUSE en–es dictionary in source word order, fixed function-word glosses, unknown words copied through.

In [ ]:
!wget -q -nc https://dl.fbaipublicfiles.com/arrival/dictionaries/en-es.txt

In [ ]:
# MUSE en-es lacks many function words; fixed glosses keep the baseline deterministic
FUNCTION_WORDS = {
    "a": "un", "an": "un", "the": "el", "of": "de", "to": "a", "in": "en", "into": "en", "on": "en",
    "at": "en", "for": "para", "by": "por", "with": "con", "from": "de", "about": "sobre", "as": "como",
    "and": "y", "or": "o", "but": "pero", "if": "si", "than": "que", "that": "que", "because": "porque",
    "is": "es", "are": "son", "was": "fue", "were": "fueron", "be": "ser", "been": "sido", "being": "siendo",
    "has": "ha", "have": "han", "had": "había", "will": "va", "would": "haría", "can": "puede",
    "could": "podría", "should": "debería", "may": "puede", "might": "podría", "must": "debe",
    "do": "hace", "does": "hace", "did": "hizo", "not": "no", "no": "no",
    "i": "yo", "you": "tú", "he": "él", "she": "ella", "it": "ello", "we": "nosotros", "they": "ellos",
    "me": "me", "him": "lo", "her": "su", "us": "nos", "them": "los",
    "my": "mi", "your": "tu", "his": "su", "its": "su", "our": "nuestro", "their": "su",
    "this": "este", "these": "estos", "those": "esos", "which": "cual", "who": "quien", "what": "qué",
    "up": "arriba", "out": "fuera", "over": "sobre", "after": "después", "before": "antes",
    "there": "allí", "here": "aquí", "also": "también", "very": "muy", "more": "más", "most": "más",
}
SUFFIXES = [("ies", "y"), ("ing", ""), ("ed", ""), ("es", ""), ("s", ""), ("ly", "")]
WORD = re.compile(r"([A-Za-z]+(?:'[A-Za-z]+)?)")


def load_muse(path):
    d = {}
    for line in open(path, encoding="utf-8"):
        parts = line.split()
        if len(parts) != 2:
            continue
        s, t = parts
        if s not in d or (d[s] == s and t != s):   # prefer first non-identity candidate
            d[s] = t
    d.update(FUNCTION_WORDS)
    return d


def gloss_word(w, d):
    lw = w.lower()
    if lw.endswith("'s"):                       # possessive: keep the noun, drop the clitic
        lw = lw[:-2]
    cands = [lw] + [lw[:-len(a)] + b for a, b in SUFFIXES if lw.endswith(a) and len(lw) - len(a) >= 3]
    for c in cands:
        if c in d:
            t = d[c]
            return (t.capitalize() if w[0].isupper() else t), True
    return w, False                             # OOV (names etc.): copy through


def gloss(text, d):
    # Word-for-word gloss in source order; punctuation and spacing kept as-is. Returns (gloss, coverage).
    out, hit, n = [], 0, 0
    for chunk in text.split():
        pieces = WORD.split(chunk)               # odd indices are words
        for i in range(1, len(pieces), 2):
            pieces[i], ok = gloss_word(pieces[i], d)
            n += 1
            hit += ok
        out.append("".join(pieces))
    return " ".join(out), (hit / n if n else 0.0)


if "es" in LANGS:
    muse = load_muse("en-es.txt")
    items["G_es"], items["gloss_cov_es"] = zip(*[gloss(s, muse) for s in items.S])
    print("gloss coverage median/min:", round(items.gloss_cov_es.median(), 3), round(items.gloss_cov_es.min(), 3))

## 3 · Prompts
Pilot wording, with the target language as a parameter. F2 no longer says "news" because other domains are not news.

In [ ]:
LANG_NAME = {"es": "Spanish", "zh": "Simplified Chinese"}

def prompts_for(lang):
    name = LANG_NAME[lang]
    only = f" Output only the {name} text, with no notes, quotes, or explanations."
    return {
        "L":  f"Translate the following English text into {name} word-for-word. Preserve the source "
              f"word order, sentence boundaries and structure as closely as {name} grammar allows. "
              "Do not rephrase, merge, split, or reorder anything." + only,
        "F":  f"Translate the following English text into {name} naturally and idiomatically, as a "
              f"skilled professional translator would for a {name}-speaking readership." + only,
        "F2": f"Render this English text in fluent, natural {name}, the way an experienced "
              "professional translator would." + only,
        "back": f"Translate the following {name} text into English. Output only the translation, with no notes.",
    }

json.dump({"vllm": vllm.__version__, "models": MODELS, "dtype": DTYPE, "langs": LANGS, "domains": DOMAINS,
           "max_w": MAX_W, "smoke_n": SMOKE_N, "max_model_len": MAX_MODEL_LEN,
           "prompts": {l: prompts_for(l) for l in LANGS}},
          open(f"{OUT_DIR}/run_config.json", "w"), indent=1, ensure_ascii=False)

## 4 · Generate
Greedy decoding. Each prompt gets `max_tokens = 2 × input tokens + 64`, capped by the context length. Outputs that hit the cap are saved with `finish = "length"` and flagged later.

In [ ]:
from vllm import LLM, SamplingParams

STORE = f"{OUT_DIR}/generations.jsonl"

def load_store():
    recs = {}
    if os.path.exists(STORE):
        for line in open(STORE, encoding="utf-8"):
            r = json.loads(line)
            recs[(r["model"], r["lang"], r["cond"], r["item_id"])] = r
    return recs

def chat_prompt(tok, system, text):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": text}]
    try:
        return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    except Exception:                            # templates without a system role: put the instruction in the user turn
        return tok.apply_chat_template([{"role": "user", "content": system + "\n\n" + text}],
                                       tokenize=False, add_generation_prompt=True)

def run(llm, model, lang, cond, system, ids, texts):
    # Translate texts under one condition, skipping items already in the store.
    tok = llm.get_tokenizer()
    done = load_store()
    todo = [(i, t) for i, t in zip(ids, texts) if (model, lang, cond, i) not in done]
    print(f"{cond}: {len(ids) - len(todo)} done, {len(todo)} to go")
    t0, n_out = time.time(), 0
    for c in range(0, len(todo), CHUNK):
        batch = todo[c:c + CHUNK]
        prompts = [chat_prompt(tok, system, t) for _, t in batch]
        n_text = [len(tok.encode(t, add_special_tokens=False)) for _, t in batch]
        n_prompt = [len(tok.encode(p, add_special_tokens=False)) for p in prompts]
        params = [SamplingParams(temperature=0, max_tokens=max(16, min(2 * a + 64, MAX_MODEL_LEN - b - 8)))
                  for a, b in zip(n_text, n_prompt)]
        outs = llm.generate(prompts, params, use_tqdm=False)
        with open(STORE, "a", encoding="utf-8") as f:
            for (i, _), o, a in zip(batch, outs, n_text):
                g = o.outputs[0]
                text = re.sub(r"<think>.*?</think>", "", g.text, flags=re.S).strip()
                f.write(json.dumps({"model": model, "lang": lang, "cond": cond, "item_id": i,
                                    "text": text, "finish": g.finish_reason,
                                    "n_in": a, "n_out": len(g.token_ids)}, ensure_ascii=False) + "\n")
                n_out += len(g.token_ids)
            f.flush(); os.fsync(f.fileno())
        k = min(c + CHUNK, len(todo)); el = time.time() - t0
        print(f"  {k}/{len(todo)}  {n_out / el:.0f} tok/s  ETA {(len(todo) - k) * el / k / 60:.0f} min")


for model in MODELS:
    llm = LLM(model=model, dtype=DTYPE, max_model_len=MAX_MODEL_LEN,
              gpu_memory_utilization=GPU_MEM, seed=SEED)
    for lang in LANGS:
        P = prompts_for(lang)
        ids, S = items.item_id.tolist(), items.S.tolist()
        noise = items[items.item_id.isin(noise_ids)]
        conds = ["F"] + (["L"] if RUN_LITERAL else [])
        for c in conds:
            run(llm, model, lang, c, P[c], ids, S)
        run(llm, model, lang, "F2", P["F2"], noise.item_id.tolist(), noise.S.tolist())
        store = load_store()
        for c in conds:                                   # T1: back to English
            fwd = [(i, store[k]["text"]) for i in ids if (k := (model, lang, c, i)) in store and store[k]["text"]]
            run(llm, model, lang, "B_" + c, P["back"], [i for i, _ in fwd], [t for _, t in fwd])
    del llm; gc.collect(); torch.cuda.empty_cache()     # if the next model OOMs: restart and run it alone

## 5 · Assemble + flag broken generations
One wide CSV per language × model. Flagged rows stay in the CSV (`bad=True`) with a reason; the analysis excludes them.
- **S1** (L/F/F2): wrong script, English left untranslated, truncated, empty.
- **T1** (B_*): wrong script, target language left untranslated, truncated, empty.

In [ ]:
has_cjk = lambda s: bool(re.search(r"[぀-ヿ一-鿿가-힯]", str(s)))
cjk_share = lambda s: len(re.findall(r"[一-鿿]", str(s))) / max(1, len(str(s).replace(" ", "")))
rate = lambda pat: (lambda s: len(re.findall(pat, str(s), re.I)) / max(1, len(str(s).split())))
en_rate = rate(r"\b(the|and|of|was|is|that|with)\b")
es_rate = rate(r"\b(el|los|las|del|que|y|una|por|con|es)\b")

def problems(text, finish, lang, side):
    out = []
    if text != text or not str(text).strip():
        return ["empty"]
    if finish == "length":
        out.append("truncated")
    if side == "S1":
        if (lang == "es" and has_cjk(text)) or (lang == "zh" and cjk_share(text) < 0.3):
            out.append("script")
        if en_rate(text) > 0.05:
            out.append("english")
    else:
        if has_cjk(text):
            out.append("script")
        if lang == "es" and es_rate(text) > 0.08:
            out.append("spanish")
    return out

recs = pd.DataFrame(load_store().values())
for (model, lang), g in recs.groupby(["model", "lang"]):
    wide = items.copy()
    if lang == "es":
        wide = wide.rename(columns={"G_es": "G", "gloss_cov_es": "gloss_cov"})
    wide = wide.drop(columns=[c for c in wide if c.endswith("_es")], errors="ignore")
    wide["bad_reason"] = ""
    for cond, rows in g.groupby("cond"):
        m = rows.set_index("item_id")
        wide[cond] = wide.item_id.map(m.text)
        wide[f"finish_{cond}"] = wide.item_id.map(m.finish)
        side = "T1" if cond.startswith("B_") else "S1"
        has = wide[cond].notna()
        wide.loc[has, "bad_reason"] += [" ".join(f"{cond}:{p}" for p in problems(t, f, lang, side)) + " "
                                        for t, f in zip(wide.loc[has, cond], wide.loc[has, f"finish_{cond}"])]
    wide["bad_reason"] = wide.bad_reason.str.split().str.join(" ")
    wide["bad"] = wide.bad_reason != ""
    path = f"{OUT_DIR}/chain_{lang}_{model.split('/')[-1]}{'_smoke' if SMOKE_N else ''}.csv"
    wide.to_csv(path, index=False)
    print(f"{model} · {lang}: {len(wide)} items, {wide.bad.sum()} flagged → {path}")
    display(wide.groupby("source").bad.agg(["sum", "mean"]).round(3))
    display(wide.loc[wide.bad, ["item_id", "words", "bad_reason"]].head(20))

In [ ]:
for _, r in wide.head(2).iterrows():
    for c in ["S", "G", "L", "F", "F2", "B_L", "B_F"]:
        if c in r and isinstance(r[c], str):
            print(f"{c:4s}:", r[c][:220])
    print("-" * 80)